# Qwen3-Reranker-0.6B: fine-tuning на 2×Tesla T4

Notebook обучает reranker на ручной разметке одинаковых товарных пар.
Он использует DDP — отдельный процесс и копию модели на каждой T4 — и
выполняет валидацию один раз, только после всех эпох.

Входной приватный Dataset: `alexproger23/product-matching-qwen-training`. В нём находятся два
исходных parquet-файла и ровно та версия `src/` и training scripts, для
которой создан notebook. Kaggle может смонтировать code bundle как ZIP
или как автоматически распакованный каталог; оба варианта проверяются.
SHA-256 исходного code bundle:
`b03c95da5ce4ed7136c15d614443d34825a85290eef085e8e48bb9b95c47aa51`.

Основные параметры запуска находятся в ячейке `TRAIN_CONFIG`. Значение
`batch_size=32` задаётся на одну GPU, поэтому эффективный batch для двух
T4 равен 64.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
TEMP_ROOT = Path("/kaggle/temp/product_matching_training")
PROJECT_ROOT = WORKING_ROOT / "product_matching"
OUTPUT_DIR = WORKING_ROOT / "qwen_products_lora"
PREPARED_DIR = TEMP_ROOT / "prepared"
TOKEN_CACHE_DIR = TEMP_ROOT / "token_cache"
TRAIN_LOG = WORKING_ROOT / "qwen_training.log"
EXPECTED_BUNDLE_SHA256 = 'b03c95da5ce4ed7136c15d614443d34825a85290eef085e8e48bb9b95c47aa51'

def exactly_one(filename):
    candidates = list(INPUT_ROOT.glob(f"**/{filename}"))
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename!r} in attached datasets, "
            f"found {candidates}"
        )
    return candidates[0]

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

items_path = exactly_one("items_human.parquet")
matches_path = exactly_one("matches.parquet")
manifest_path = exactly_one('training_bundle_manifest.json')
attached_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
bundle_candidates = list(INPUT_ROOT.glob("**/product_matching_training_code.zip"))
bundle_candidates.extend(
    path
    for path in INPUT_ROOT.glob("**/product_matching_training_code")
    if path.is_dir()
)
if len(bundle_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one source bundle as ZIP or expanded directory, "
        f"found {bundle_candidates}"
    )
bundle_path = bundle_candidates[0]

TEMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if bundle_path.is_file():
    actual_bundle_hash = file_sha256(bundle_path)
    if actual_bundle_hash != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError(
            "Attached source bundle does not match this notebook: "
            f"expected {EXPECTED_BUNDLE_SHA256}, got {actual_bundle_hash}"
        )
    with zipfile.ZipFile(bundle_path) as archive:
        for member in archive.namelist():
            member_path = PurePosixPath(member)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"Unsafe code-bundle member: {member!r}")
        archive.extractall(PROJECT_ROOT)
    bundle_layout = "zip"
else:
    shutil.copytree(bundle_path, PROJECT_ROOT, dirs_exist_ok=True)
    bundle_layout = "expanded_directory"

expected_source_manifest = attached_manifest["code_bundle"]["source"]
bundled_source_manifest = json.loads(
    (PROJECT_ROOT / "source_manifest.json").read_text(encoding="utf-8")
)
if bundled_source_manifest != expected_source_manifest:
    raise RuntimeError("Expanded source_manifest.json does not match dataset manifest")
for relative_name, expected in expected_source_manifest["files"].items():
    relative_path = PurePosixPath(relative_name)
    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise RuntimeError(f"Unsafe source manifest path: {relative_name!r}")
    source_path = PROJECT_ROOT.joinpath(*relative_path.parts)
    if not source_path.is_file():
        raise RuntimeError(f"Bundled source file is missing: {relative_name}")
    actual_hash = file_sha256(source_path)
    if actual_hash != expected["sha256"]:
        raise RuntimeError(
            f"Source hash mismatch for {relative_name}: "
            f"expected {expected['sha256']}, got {actual_hash}"
        )

print(f"items:   {items_path} ({items_path.stat().st_size / 2**20:.1f} MiB)")
print(f"matches: {matches_path} ({matches_path.stat().st_size / 2**20:.1f} MiB)")
print(
    f"source:  {bundle_path} (layout={bundle_layout}, "
    f"fingerprint={EXPECTED_BUNDLE_SHA256})"
)
print(subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True).stdout)

## Зависимости и подготовка данных

PyTorch из Kaggle image сохраняется, если он удовлетворяет диапазону в
`requirements-gpu.txt`; доустанавливаются Transformers и PEFT. Подготовленные
parquet-файлы и mmap token cache пишутся в `/kaggle/temp`, поэтому они не
раздувают скачиваемый output notebook.

In [ ]:
requirements_path = PROJECT_ROOT / "requirements-gpu.txt"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--upgrade-strategy",
        "only-if-needed",
        "-r",
        str(requirements_path),
    ],
    check=True,
)

prepare_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/prepare_human_data.py"),
    "--items",
    str(items_path),
    "--matches",
    str(matches_path),
    "--output-dir",
    str(PREPARED_DIR),
]
print("$", " ".join(prepare_command), flush=True)
subprocess.run(prepare_command, check=True, cwd=PROJECT_ROOT)

## Параметры обучения

`batch_size` и `eval_batch_size` указаны на один DDP process/GPU.
Четыре DataLoader worker'а работают параллельно: по два рядом с каждой T4.
`attention_mlp` обучает LoRA не только на attention projections, но и на MLP.

In [ ]:
TRAIN_CONFIG = {
    "model": "Qwen/Qwen3-Reranker-0.6B",
    "epochs": 2,
    "batch_size": 32,
    "eval_batch_size": 32,
    "gradient_accumulation": 1,
    "max_length": 384,
    "learning_rate": 2e-4,
    "lora_rank": 16,
    "lora_targets": "attention_mlp",
    "sampling": "category_label",
    "dataloader_workers": 2,
    "prefetch_factor": 2,
    "tokenization_batch_size": 512,
    "log_every": 25,
    "seed": 42,
}
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))

## DDP fine-tuning

`torch.distributed.run` создаёт два процесса, поэтому обе T4 вычисляют
разные части глобального batch. Вывод одновременно виден в notebook и
сохраняется в `qwen_training.log`. Валидация запускается training script
один раз — после завершения последней эпохи.

In [ ]:
train_command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    str(PROJECT_ROOT / "scripts/train_qwen_names.py"),
    "--prepared-dir",
    str(PREPARED_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--token-cache-dir",
    str(TOKEN_CACHE_DIR),
    "--model",
    TRAIN_CONFIG["model"],
    "--epochs",
    str(TRAIN_CONFIG["epochs"]),
    "--batch-size",
    str(TRAIN_CONFIG["batch_size"]),
    "--eval-batch-size",
    str(TRAIN_CONFIG["eval_batch_size"]),
    "--gradient-accumulation",
    str(TRAIN_CONFIG["gradient_accumulation"]),
    "--max-length",
    str(TRAIN_CONFIG["max_length"]),
    "--learning-rate",
    str(TRAIN_CONFIG["learning_rate"]),
    "--lora-rank",
    str(TRAIN_CONFIG["lora_rank"]),
    "--lora-targets",
    TRAIN_CONFIG["lora_targets"],
    "--sampling",
    TRAIN_CONFIG["sampling"],
    "--dataloader-workers",
    str(TRAIN_CONFIG["dataloader_workers"]),
    "--prefetch-factor",
    str(TRAIN_CONFIG["prefetch_factor"]),
    "--tokenization-batch-size",
    str(TRAIN_CONFIG["tokenization_batch_size"]),
    "--log-every",
    str(TRAIN_CONFIG["log_every"]),
    "--seed",
    str(TRAIN_CONFIG["seed"]),
]
training_environment = os.environ.copy()
training_environment.update(
    {
        "OMP_NUM_THREADS": "2",
        "TOKENIZERS_PARALLELISM": "false",
        "NCCL_DEBUG": "WARN",
        "PYTHONUNBUFFERED": "1",
    }
)
print("$", " ".join(train_command), flush=True)
training_started = time.perf_counter()
with TRAIN_LOG.open("w", encoding="utf-8", buffering=1) as log_file:
    process = subprocess.Popen(
        train_command,
        cwd=PROJECT_ROOT,
        env=training_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
    return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, train_command)
training_wall_seconds = time.perf_counter() - training_started
print(f"Training process finished in {training_wall_seconds / 3600:.2f} hours")

## Итоговый отчёт и сохраняемые outputs

In [ ]:
report_path = OUTPUT_DIR / "training_report.json"
if not report_path.is_file():
    raise RuntimeError(f"Training finished without report: {report_path}")
report = json.loads(report_path.read_text(encoding="utf-8"))
completion = {
    "status": "complete",
    "code_bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "training_wall_seconds": training_wall_seconds,
    "training_report": report,
}
completion_path = WORKING_ROOT / "notebook_completed.json"
completion_path.write_text(
    json.dumps(completion, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(json.dumps(completion, ensure_ascii=False, indent=2, default=str))
print("Saved outputs:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(WORKING_ROOT)}: {path.stat().st_size / 2**20:.2f} MiB")
print(f"  {TRAIN_LOG.name}: {TRAIN_LOG.stat().st_size / 2**20:.2f} MiB")
print(f"  {completion_path.name}: {completion_path.stat().st_size / 2**20:.2f} MiB")